In [1]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip ml-latest-small.zip

--2026-09-25 11:08:20--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  2.63MB/s    in 0.4s    

2026-09-25 11:08:22 (2.63 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


In [2]:
import pandas as pd

movies = pd.read_csv('ml-latest-small/movies.csv')
ratings = pd.read_csv('ml-latest-small/ratings.csv')

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
print("Number of movies:", movies.shape[0])
print("Number of ratings:", ratings.shape[0])

# Most common genres
from collections import Counter
genre_counts = Counter()
for g in movies['genres'].str.split('|'):
    genre_counts.update(g)
print(genre_counts.most_common(10))

Number of movies: 9742
Number of ratings: 100836
[('Drama', 4361), ('Comedy', 3756), ('Thriller', 1894), ('Action', 1828), ('Romance', 1596), ('Adventure', 1263), ('Crime', 1199), ('Sci-Fi', 980), ('Horror', 978), ('Fantasy', 779)]


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Replace '|' with spaces so genres are treated as separate words
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

# Convert genre text into numeric vectors
tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies['genres_clean'])

print(genre_matrix.shape)  # (num_movies, num_unique_genre_words)

(9742, 24)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(genre_matrix)
print(similarity.shape)  # (num_movies, num_movies) - similarity score between every pair

(9742, 9742)


In [6]:
# Create a lookup: movie title -> index
indices = pd.Series(movies.index, index=movies['title'])

def recommend(title, n=5):
    if title not in indices:
        return f"'{title}' not found in dataset. Try checking exact spelling."

    idx = indices[title]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = scores[1:n+1]  # skip the movie itself

    movie_indices = [i[0] for i in scores]
    return movies['title'].iloc[movie_indices]

# Test it
recommend('Toy Story (1995)')

,title
1706,Antz (1998)
2355,Toy Story 2 (1999)
2809,"Adventures of Rocky and Bullwinkle, The (2000)"
3000,"Emperor's New Groove, The (2000)"
3568,"Monsters, Inc. (2001)"


In [7]:
print(recommend('Jumanji (1995)'))
print(recommend('Batman Forever (1995)'))

53             Indian in the Cupboard, The (1995)
109             NeverEnding Story III, The (1994)
767               Escape to Witch Mountain (1975)
1514    Darby O'Gill and the Little People (1959)
1556                          Return to Oz (1985)
Name: title, dtype: object
126                      Batman Forever (1995)
3511    It's a Mad, Mad, Mad, Mad World (1963)
4071                              I Spy (2002)
4726                    Nothing to Lose (1997)
6274                      Crime Busters (1977)
Name: title, dtype: object


In [8]:
avg_ratings = ratings.groupby('movieId')['rating'].mean()
movies['avg_rating'] = movies['movieId'].map(avg_ratings)

def recommend_v2(title, n=5):
    if title not in indices:
        return f"'{title}' not found."
    idx = indices[title]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:20]  # top 20 similar
    movie_indices = [i[0] for i in scores]

    result = movies.iloc[movie_indices][['title', 'avg_rating']]
    return result.sort_values('avg_rating', ascending=False).head(n)

recommend_v2('Toy Story (1995)')

,title,avg_rating
7760,Asterix and the Vikings (Astérix et les Viking...,5.000000
3568,"Monsters, Inc. (2001)",3.871212
2355,Toy Story 2 (1999),3.860825
8900,Inside Out (2015),3.813953
1505,"Black Cauldron, The (1985)",3.750000
